$e = mc^2$  
$$E = mc^2$$  
$$x^2$$  
$$x_i$$  
$$\frac{a}{b}$$  
$$e^{i\pi}$$  
$$\theta, \lambda, \pi, \alpha, \beta $$  
$$
A = \begin{bmatrix}
a & b \\
c & d
\end{bmatrix}
$$  
$$
\begin{align}
f(x) &= (x + 1)^2 \\
     &= x^2 + 2x + 1
\end{align}
$$


In [1]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped

polars.config.Config

In [2]:
path = "/Users/oceansxyz/alphanode-metal/logs/matrix/matrix_20260619_085714.jsonl"

#sample : {"kind":"matrix","ts":1781859435549,"buy_anchor_alpha":0.9981485732,"sell_anchor_alpha":0.9969508888,"bps_buy_anchor":-18.5143,"bps_sell_anchor":-30.4911,"b0":62579.9900000000,"a0":62580.0000000000,"b1":54531.0100000000,"a1":54625.4400000000,"b2":1.1472000000,"a2":1.1474000000}

In [3]:
# Raw matrix math log — config row then alpha snapshots
matrix_row_schema = pl.Struct([
    pl.Field("kind", pl.String),
    pl.Field("ts", pl.Int64),
    pl.Field("buy_anchor_alpha", pl.Float64),
    pl.Field("sell_anchor_alpha", pl.Float64),
    pl.Field("bps_buy_anchor", pl.Float64),
    pl.Field("bps_sell_anchor", pl.Float64),
    pl.Field("b0", pl.Float64),
    pl.Field("a0", pl.Float64),
    pl.Field("b1", pl.Float64),
    pl.Field("a1", pl.Float64),
    pl.Field("b2", pl.Float64),
    pl.Field("a2", pl.Float64),
])

with open(path) as f:
    config_json = f.readline().strip()
    print("Config Raw Line:", config_json)
    df_matrix = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=matrix_row_schema))
        .unnest("json_str")
        .filter(col("kind") == "matrix")
    )



Config Raw Line: {"kind":"config","live":false,"tickers":["BTCUSDT","BTCEURI","EURIUSDT"],"maxNotionalQty":5.0000}


$$m01 = BTC^{-1} \cdot USDT  $$
$$m10 = BTC \cdot USDT^{-1}  $$  
$$m12 = BTCEURI  $$  
$$m21 = BTCEURI  $$  
$$m20 = EURIUSDT $$  
$$m02 = EURIUSDT $$  
  
    
$$\alpha_{cw}$$  
$$\alpha_{ccw}$$


In [4]:
df_matrix.head()

kind,ts,buy_anchor_alpha,sell_anchor_alpha,bps_buy_anchor,bps_sell_anchor,b0,a0,b1,a1,b2,a2
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""matrix""",1781859435549,0.998149,0.996951,-18.5143,-30.4911,62579.99,62580.0,54531.01,54625.44,1.1472,1.1474
"""matrix""",1781859435553,0.998149,0.996951,-18.5143,-30.4911,62579.99,62580.0,54531.01,54625.44,1.1472,1.1474
"""matrix""",1781859436242,0.998149,0.995645,-18.5143,-43.5506,62579.99,62580.0,54531.01,54697.09,1.1472,1.1474
"""matrix""",1781859436242,0.998149,0.996951,-18.5143,-30.4929,62579.99,62580.0,54531.01,54625.45,1.1472,1.1474
"""matrix""",1781859436545,0.998149,0.996951,-18.5143,-30.4929,62579.99,62580.0,54531.01,54625.45,1.1472,1.1474


In [5]:
print(df_matrix.columns)

['kind', 'ts', 'buy_anchor_alpha', 'sell_anchor_alpha', 'bps_buy_anchor', 'bps_sell_anchor', 'b0', 'a0', 'b1', 'a1', 'b2', 'a2']


#### <p align="center">**_BUY on Anchor SELL on Target_**</p>  

1. BUY  $y$ Anchor Alt Coin on Anchor Venue (BTCUSDT)  @ask  
    * SELL $x_0$ Anchor Stable Coin on Anchor Venue  
    * $y = x_0 / a_0    $=>$  $[BTC] = [USDT] / [USDT][BTC^{-1}]   $
2. SELL $y$ Anchor Alt Coin on Target Venue (BTCEURI)  @bid
    * BUY $z$ Target Stable Coin on Target Venue  
    * $z = y * b_1 $=>$  $[EURI] = [BTC]*[EURI][BTC^{-1}]   $
 
3. SELL $z$ Target Stable Coin on Bridge Venue (EURIUSDT) @bid  
    * BUY $x_1$ Anchor Stable Coin on Bridge Venue 
    * $x_1 = z * b_2$


    $$\alpha = x_1 / x_0 $$
    $$\alpha_{bps} = (1-\alpha)*(1e+4) $$


In [6]:
buy_anchor_alpha_df = df_matrix.select(['ts', 'buy_anchor_alpha', 'bps_buy_anchor', 
'b0', 'a0', 'b1', 'a1', 'b2', 'a2'])



buy_anchor_alpha_df.head()

ts,buy_anchor_alpha,bps_buy_anchor,b0,a0,b1,a1,b2,a2
i64,f64,f64,f64,f64,f64,f64,f64,f64
1781859435549,0.998149,-18.5143,62579.99,62580.0,54531.01,54625.44,1.1472,1.1474
1781859435553,0.998149,-18.5143,62579.99,62580.0,54531.01,54625.44,1.1472,1.1474
1781859436242,0.998149,-18.5143,62579.99,62580.0,54531.01,54697.09,1.1472,1.1474
1781859436242,0.998149,-18.5143,62579.99,62580.0,54531.01,54625.45,1.1472,1.1474
1781859436545,0.998149,-18.5143,62579.99,62580.0,54531.01,54625.45,1.1472,1.1474


#### <p align="center">**SELL on Anchor BUY on Target_**</p>  

1. SELL $y$ Anchor Alt Coin on Anchor Venue (BTCUSDT)  @bid  
    * RECV $x_0$ Anchor Stable Coin on Anchor Venue  
    * $y = x_0 / b_0    $=>$  $[BTC] = [USDT] / [USDT][BTC^{-1}]   $
2. BUY $y$ Anchor Alt Coin on Target Venue (BTCEURI)  @ask
    * SEND $z_0$ Target Stable Coin on Target Venue  
    * $z_0 = y * a_1 $=>$  $[EURI] = [BTC]*[EURI][BTC^{-1}]   $
 
3. BUY $z_1$ Target Stable Coin on Bridge Venue (EURIUSDT) @bid  
    * BUY $x_1$ Anchor Stable Coin on Bridge Venue 
    * $x_1 = z * b_2$


    $$\alpha = x_1 / x_0 $$
    $$\alpha_{bps} = (1-\alpha)*(1e+4) $$